# Setup

## Imports

In [1]:
%load_ext autoreload
%autoreload 2

## IMPORTS
import os
from typing import Literal

import geopandas as gpd
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from shapely.geometry import Point, box
from shapely.ops import unary_union

import src.data_filepaths as data_filepaths
import src.utils.loaders as loaders
import src.utils.vis.colors_utils as colors_utils
from src.utils.df_tools import map_values_b_to_bp
from src.utils.loaders import process_loaded_building_df
from src.utils.vis.map import CategoricalColorMap, MapExplorer
from src.utils.vis.map_special import (
    build_map_building_basic,
    build_map_point_floorspace_demand_type,
    build_map_point_numeric_and_category
)

# configs
sns.set_theme()

## Buildings Data Loading

In [2]:
b_gdf = process_loaded_building_df(gpd.read_parquet(data_filepaths.OVERTURE_BUILDING))
bp_gdf = process_loaded_building_df(
    gpd.read_parquet(data_filepaths.OVERTURE_BUILDING_PART)
)

osm_landuse_gdf = loaders.load_osm_landuse_1_geojson_to_gdf(data_filepaths.OSM_landuse)
mp_landuse_gdf = loaders.load_masterplan_landuse_geojson_to_gdf(
    data_filepaths.MASTERPLAN_LANDUSE
)

In [3]:
def merge_with_landuse(
    b_gdf: gpd.GeoDataFrame,
    # bp_gdf: gpd.GeoDataFrame,
):
    """
    Returns `buildings_osm_merge`, `buildings_mp_merge`
    """

    buildings_osm_merge = gpd.sjoin(
        b_gdf[["centroid"]].set_geometry("centroid"),
        osm_landuse_gdf,
        how="left",
        predicate="intersects",
    )
    # deduplicate buildings having more than one associated plot/area
    buildings_osm_merge = buildings_osm_merge[
        ~buildings_osm_merge.index.duplicated(keep="first")
    ]

    buildings_osm_merge = buildings_osm_merge.drop(columns="index_right")

    buildings_mp_merge = gpd.sjoin(
        b_gdf[["centroid"]].set_geometry("centroid"),
        mp_landuse_gdf,
        how="left",
        predicate="intersects",
    )
    # deduplicate buildings having more than one associated plot/area
    buildings_mp_merge = buildings_mp_merge[
        ~buildings_mp_merge.index.duplicated(keep="first")
    ]

    buildings_mp_merge["landuse"].value_counts(dropna=False)

    # fix bool columns, because pandas reasons...?
    buildings_mp_merge["is_sdp"] = buildings_mp_merge["is_sdp"].astype(bool)
    buildings_mp_merge["is_landed"] = buildings_mp_merge["is_landed"].astype(bool)

    # no assigning to b_gdf is being done
    return buildings_osm_merge, buildings_mp_merge

In [4]:
buildings_osm_merge, buildings_mp_merge = merge_with_landuse(b_gdf)

In [5]:
buildings_mp_merge

,centroid,OBJECTID,landuse,gpr,white_q_max,gpr_b_min,is_landed,is_sdp,color
id,,,,,,,,,
e71fc534-b314-449f-9ec9-1e93ab103733,POINT (5877.918 22344.533),619836.0,PORT / AIRPORT,0.0,NaN,NaN,False,False,"(0.39215686274509803, 0.40784313725490196, 0.4..."
af894262-c019-4f4d-9b7c-71c934d25f9e,POINT (6069.226 21635.181),619284.0,RESERVE SITE,0.0,NaN,NaN,False,False,"(0.8980392156862745, 0.9058823529411765, 0.396..."
4a403634-2a1c-470d-8272-ed1283c02a6c,POINT (6079.727 21628.12),619284.0,RESERVE SITE,0.0,NaN,NaN,False,False,"(0.8980392156862745, 0.9058823529411765, 0.396..."
0c34c5e2-67c0-4593-ba6d-90a5f838fe8a,POINT (6096.4 21622.932),619284.0,RESERVE SITE,0.0,NaN,NaN,False,False,"(0.8980392156862745, 0.9058823529411765, 0.396..."
0a85c6b1-0f74-4cb2-9a27-30b12e3a9ab7,POINT (6097.073 21637.741),619284.0,RESERVE SITE,0.0,NaN,NaN,False,False,"(0.8980392156862745, 0.9058823529411765, 0.396..."
...,...,...,...,...,...,...,...,...,...
0e444aaa-d16f-4868-a52e-ab2c6f853a7c,POINT (52524.382 39530.303),618565.0,RESERVE SITE,0.0,NaN,NaN,False,False,"(0.8980392156862745, 0.9058823529411765, 0.396..."
24bb2058-f406-4cf4-b7db-91d0420c01cc,POINT (52508.234 39531.899),618565.0,RESERVE SITE,0.0,NaN,NaN,False,False,"(0.8980392156862745, 0.9058823529411765, 0.396..."
1abaf7e4-34eb-42c9-bb37-e625218b360f,POINT (52072.806 39434.689),618565.0,RESERVE SITE,0.0,NaN,NaN,False,False,"(0.8980392156862745, 0.9058823529411765, 0.396..."


## Select Healthcare Zones & buildings

In [6]:
buildings_mp_merge.columns

Index(['centroid', 'OBJECTID', 'landuse', 'gpr', 'white_q_max', 'gpr_b_min',
       'is_landed', 'is_sdp', 'color'],
      dtype='str')

In [7]:
mp_landuse_gdf

,landuse,gpr,white_q_max,gpr_b_min,geometry,is_landed,is_sdp,color
OBJECTID,,,,,,,,
692393,RESIDENTIAL,0.0,NaN,NaN,"POLYGON ((34229.307 31524.984, 34229.261 31525...",True,False,"(0.9568627450980393, 0.7294117647058823, 0.509..."
692394,PARK,0.0,NaN,NaN,"POLYGON ((36575.285 33625.177, 36580.052 33632...",False,False,"(0.0, 0.6235294117647059, 0.20392156862745098,..."
692395,PARK,0.0,NaN,NaN,"POLYGON ((37081.363 34501.719, 37072.045 34510...",False,False,"(0.0, 0.6235294117647059, 0.20392156862745098,..."
692396,RESIDENTIAL,0.0,NaN,NaN,"POLYGON ((40358.645 35118.256, 40350.281 35130...",True,False,"(0.9568627450980393, 0.7294117647058823, 0.509..."
692397,RESIDENTIAL,1.4,NaN,NaN,"POLYGON ((36226.649 34002.608, 36226.842 34011...",False,False,"(0.9568627450980393, 0.7294117647058823, 0.509..."
...,...,...,...,...,...,...,...,...
689475,RESIDENTIAL,0.0,NaN,NaN,"POLYGON ((32228.224 34727.994, 32225.476 34736...",True,False,"(0.9568627450980393, 0.7294117647058823, 0.509..."
689476,RESIDENTIAL,0.0,NaN,NaN,"POLYGON ((32350.318 34746.584, 32350.12 34747....",True,False,"(0.9568627450980393, 0.7294117647058823, 0.509..."
689477,ROAD,NaN,NaN,NaN,"POLYGON ((35520.071 33159.183, 35510.027 33210...",False,False,"(0.00784313725490196, 0.08235294117647059, 0.0..."


In [8]:
from src.utils.building_tools import merge_building_parts_into_buildings


def select_healthcare_zones_buildings(
    b_gdf: gpd.GeoDataFrame,
    bp_gdf: gpd.GeoDataFrame,
    buildings_mp_merge: gpd.GeoDataFrame,
    mp_landuse_gdf: gpd.GeoDataFrame,
    debug: bool = False,
    vis: bool = False,
):
    """
    Filters masterplan plots for those that have a decent likelihood of being currently used for healthcare purposes.

    Outputs the set of **masterplan plots** only, with no building information.

    Only (Estimated) `floorspace` is available, calculated based on the available building data
    to aid with workforce/patient distribution.

    ## Merging rules
    My empirical rules for whether or not a MP-zoned healthcare plot is actually being used for healthcare:
    * Building coverage > 20%
    * Overture building use:
        * whitelist: `subtype=medical`, `subtype=residential` (sometimes nursing homes get tagged as such)
        * blacklist: `subtype=industrial`
    * The condition is currently written to be TRUE_HEALTHCARE =
        (cov > 0.20) | (bldg tag in whitelist) & ~(bldg tag in blacklist)

    There is also a further manual removal blacklist, by MP plot ID.
    """

    b_gdf[["mp_landuse"]] = buildings_mp_merge[["landuse"]]
    b_gdf[["mp_gpr"]] = buildings_mp_merge[["gpr"]]
    b_gdf[["mp_OBJECTID"]] = buildings_mp_merge[["OBJECTID"]]

    # first select healthcare zones only, and buildings/bparts inside
    buildings_mp_merge = buildings_mp_merge.copy()[
        buildings_mp_merge["landuse"] == "HEALTH & MEDICAL CARE"
    ]
    b_gdf = b_gdf.copy()[b_gdf.index.isin(buildings_mp_merge.index)]
    bp_gdf = bp_gdf.copy()[bp_gdf["building_id"].isin(b_gdf.index)]

    # merge BP -> B
    b_gdf = merge_building_parts_into_buildings(b_gdf, bp_gdf)

    # finding converage thresholds :)
    b_mp_covered = b_gdf.groupby("mp_OBJECTID")["area"].sum()
    mp_area = mp_landuse_gdf.geometry.area
    b_gdf["b_mp_coverage_prop"] = b_gdf["mp_OBJECTID"].map(b_mp_covered / mp_area)

    if vis:
        sns.histplot(b_mp_covered / mp_area)

    # remove unlikely healthcare plots
    # see above for criterion
    health_b_gdf = b_gdf[
        (
            (b_gdf["b_mp_coverage_prop"] > 0.20)
            | (b_gdf["subtype"].isin(["residential", "medical"]))  # whitelist
        )
        & ~(
            b_gdf["subtype"].isin(["industrial"])  # blacklist
            # manually defined removal list
            # 75 pasir panjang rd
            # CCK Dorm B (1)
            # CCK Dorm B (2)
            # Old Institute of Health at Outram Road
            # Alexandra Hosp future extension
            | b_gdf["mp_OBJECTID"].isin([687394, 614573, 616545, 629680, 671871])
        )
    ]
    buildings_mp_merge = (
        buildings_mp_merge[["OBJECTID"]]
        .merge(
            health_b_gdf[["names", "area", "num_floors"]],
            left_index=True,
            right_index=True,
        )
        .merge(mp_landuse_gdf["geometry"], left_on="OBJECTID", right_index=True)
    )

    ## aggregate buildings
    # default floorcount of 1.5 used
    buildings_mp_merge["floorspace"] = buildings_mp_merge[
        "area"
    ] * buildings_mp_merge["num_floors"].clip(1.5).fillna(1.5)

    health_plots = buildings_mp_merge.groupby("OBJECTID").agg(
        {
            "names": lambda x: [
                a for a in [d["primary"] if d else "" for d in x] if a != ""
            ],
            "floorspace": "sum",
            "geometry": "first",
        }
    )
    health_plots = health_plots.set_geometry("geometry").set_crs(mp_landuse_gdf.crs)
    health_plots["centroid"] = health_plots.centroid
    health_plots["area"] = health_plots.area

    if vis:
        # return build_map_building_basic(b_gdf)
        mapper = MapExplorer(health_b_gdf)
        return mapper.build_singleprop("b_mp_coverage_prop")

    return health_plots


health_plots_gdf = select_healthcare_zones_buildings(
    b_gdf,
    bp_gdf,
    buildings_mp_merge,
    mp_landuse_gdf,
    debug=True,
    # vis=True,
)

# health_plots.head(4)
health_plots_gdf.sort_values("area", ascending=False)

7


,names,floorspace,geometry,centroid,area
OBJECTID,,,,,
662035.0,"[Ren Ci Hospital, Institute Of Mental Health]",87101.513711,"POLYGON ((33989.678 40568.846, 33987.395 40570...",POINT (33735.827 40494.807),185665.012968
656790.0,"[Service Block, Main Building, Tower Block, Ke...",240995.791972,"POLYGON ((22670.46 30693.927, 22670.285 30694....",POINT (22422.679 30743.725),165680.830591
678888.0,"[SGH Emergency/ NNI Building, Blk 1, Blk 3, Bl...",402597.095485,"POLYGON ((28345.48 29364.583, 28345.466 29364....",POINT (28153.821 29115.833),121690.291700
678590.0,"[Block 5, Block 2, Block 3B, Block 3C, Block 2...",57888.788822,"POLYGON ((24618.702 29915.011, 24618.803 29915...",POINT (24378.559 29901.867),118386.763022
640109.0,"[National Heart Centre Singapore, National Can...",404394.276426,"POLYGON ((28383.872 28919.498, 28386.576 28920...",POINT (28470.511 29120.347),94170.457794
...,...,...,...,...,...
640742.0,[],778.812833,"POLYGON ((36128.855 33246.754, 36128.441 33248...",POINT (36073.737 33240.937),789.691139
593742.0,[Econ Medicare Centre],451.197289,"POLYGON ((31905.469 36170.959, 31919.375 36204...",POINT (31900.299 36190.317),760.918540
631310.0,[],763.242446,"POLYGON ((31553.258 36318.717, 31553.255 36318...",POINT (31545.856 36318.496),751.434151


In [9]:
build_map_point_numeric_and_category(
    health_plots_gdf, numeric_col="floorspace", geometry_col="centroid"
)

# Initial Data

In [10]:
from src.data_manual.hospitals import Hospital, HOSPITALS

## Manual Data Entry

In [11]:
STATS_OVERALL = {
    "totals": {
        "workforce": 138000, # NO LONGER BEING USED
        "beds_publlc": 14333,
    }
}

STATS_GROUP = {
    "NHG": {
        "workforce": 25522,
    },
    "IHH Healthcare": {
        "workforce": 5400,
    },
}

## Process Data

should differentiate b/w primary care and nursing homes

and unknown

basically have to use names

* polyclinic - straightfwd
* care centre, hospice, care home, home

### Integrate pre-defined Hospitals into MP Zoning Data

Also sets up the df

In [12]:
def integrate_pre_defined_hospitals(
    hospitals: list[Hospital],
    health_plots_gdf: gpd.GeoDataFrame,
):
    """
    For hospitals with multiple plot_ids defined, geographically merge

    `name` for hospitals is assumed to be unique

    **Output GDF format**
    * Columns:
        * `name`: predefined name from hospitals likely take over previous gdf `names` lists that had multiple (building) names.
        The first name from the previous gdf `names` list will otherwise be taken.

    """

    # convert hospitals_dict to df
    hospitals_df = pd.DataFrame([a.dict for a in hospitals]).reset_index(
        names="_hosp_id"
    )
    # explode plot_ids
    hospitals_df = hospitals_df.explode("plot_ids", ignore_index=True)

    hospitals_merged_gdf = hospitals_df.join(health_plots_gdf, on="plot_ids")

    # group & aggregate based on hospitals
    hospitals_gdf_agg = hospitals_merged_gdf.groupby(
        "_hosp_id",  # assumption
        sort=False,
    ).agg(
        **{
            col: (col, "first")
            for col in [
                "name",
                "workforce",
                "beds",
                "inpatient",
                "outpatient",
                "group",
                "healthcare_type",
            ]
        },
        floorspace=("floorspace", "sum"),
        area=("area", "sum"),
        geometry=("geometry", unary_union),
    )

    # now handle non-predefined (other, non-hospital) healthcare plots
    health_plots_non_predef = health_plots_gdf.loc[
        ~health_plots_gdf.index.isin(set(hospitals_merged_gdf["plot_ids"].dropna()))
    ].copy()
    # make a few tweaks
    health_plots_non_predef["name"] = health_plots_non_predef["names"].apply(
        lambda x: x[0] if x else None
    )
    health_plots_non_predef = health_plots_non_predef.drop(columns=["names"])

    for col in hospitals_gdf_agg.columns:
        if col not in health_plots_non_predef.columns:
            health_plots_non_predef[col] = np.nan

    # reodder cols
    health_plots_non_predef = health_plots_non_predef.reset_index()
    # here plot IDs are also dropped... shouldn't need them anymore
    # (esp because the combined hospitals would have multiple plots, and I don't want to think abt that)
    health_plots_non_predef = health_plots_non_predef[hospitals_gdf_agg.columns]

    res_gdf = gpd.GeoDataFrame(
        pd.concat([hospitals_gdf_agg, health_plots_non_predef], ignore_index=True),
        geometry="geometry",
        crs=health_plots_gdf.crs,
    )

    # recalc centroids
    res_gdf["centroid"] = res_gdf.centroid

    return res_gdf


health_plots_full_gdf = integrate_pre_defined_hospitals(HOSPITALS, health_plots_gdf)
health_plots_full_gdf.sort_values("floorspace", ascending=False).reset_index(drop=True).head(20)

p:\Code2\1_PROJ_GROUPS\10_TrainGames\Subway Builder\Mapping\.venv\Lib\site-packages\geopandas\array.py:1770: UserWarning: CRS not set for some of the concatenation inputs. Setting output's CRS as SVY21 / Singapore TM (the single non-null crs provided).
  return GeometryArray(data, crs=_get_common_crs(to_concat))


,name,workforce,beds,inpatient,outpatient,group,healthcare_type,floorspace,area,geometry,centroid
0,Tan Tock Seng Hospital,NaN,1700.0,NaN,NaN,NHG,acute,412128.379805,138731.275296,"MULTIPOLYGON (((29590.189 33473.374, 29581.087...",POINT (29516.374 33711.208)
1,SingHealth Outram Specialist Centres,5083.0,185.0,NaN,NaN,SH,acute,404394.276426,94170.457794,"POLYGON ((28383.872 28919.498, 28386.576 28920...",POINT (28470.511 29120.347)
2,Singapore General Hospital,10725.0,1931.0,NaN,NaN,SH,acute,402597.095485,121690.291700,"POLYGON ((28345.48 29364.583, 28345.466 29364....",POINT (28153.821 29115.833)
3,Sengkang General Hospital,5979.0,1320.0,NaN,NaN,SH,acute,302192.442428,67291.431531,"POLYGON ((34572.858 41763.457, 34632.583 41742...",POINT (34608.481 41925.392)
4,Changi General Hospital,7092.0,1132.0,NaN,NaN,SH,acute,255082.967672,54002.480305,"MULTIPOLYGON (((40990.17 35744.257, 40988.746 ...",POINT (40901.362 35845.971)
5,Woodlands Hospital,NaN,1800.0,NaN,NaN,NHG,acute,245437.238286,76723.749501,"POLYGON ((23840.722 45284.235, 23834.816 45286...",POINT (23706.117 45161.249)
6,National University Hospital,10700.0,1700.0,NaN,NaN,NUHS,acute,240995.791972,165680.830591,"POLYGON ((22670.46 30693.927, 22670.285 30694....",POINT (22422.679 30743.725)
7,Ng Teng Fong General Hospital,NaN,1100.0,NaN,NaN,NUHS,acute,196179.465985,54031.899088,"MULTIPOLYGON (((18385.195 34926.313, 18384.791...",POINT (18197.418 35121.564)
8,Khoo Teck Puat Hospital,NaN,1019.0,NaN,NaN,NHG,acute,185731.906567,44214.755940,"POLYGON ((28458.873 44982.973, 28405.074 44982...",POINT (28533.401 45100.17)
9,KK Women's and Children's Hospital,5411.0,864.0,NaN,NaN,SH,acute,152371.224310,44723.295021,"POLYGON ((29612.679 32542.31, 29613.26 32544.2...",POINT (29509.22 32537.082)


In [13]:
build_map_point_numeric_and_category(
    health_plots_full_gdf, numeric_col="floorspace", geometry_col="centroid"
)

# Workforce Distribution

## Basic LinReg Redistributor

In [14]:
from sklearn.linear_model import (
    LinearRegression,
    GammaRegressor,
    PoissonRegressor,
)


def distribute_values_regr(
    X: pd.DataFrame,
    y: pd.Series,
    X_pred: pd.DataFrame,
    regr_model=GammaRegressor,
    total_val: float = None,
):
    """
    Default is sklearn's GammaRegressor (link-log)
    """
    model = regr_model()
    model.fit(X, y)

    y_pred = model.predict(X_pred)
    if not total_val:
        return y_pred

    return y_pred / np.sum(y_pred) * total_val

## Distribute Workforce based on pre-defined Grouped Stats

In [15]:
def distribute_workforce_grouped(hp_gdf: gpd.GeoDataFrame, stats_group: dict):
    """
    Distribution is done by...
    regression, beds + floorspace
    currently using Gamma Regressor (log-link)

    TODO: whether IMH needs to have a specific tweak made to suppress its bedcount influence? 
    Not done at the moment

    Args:
        hp_gdf (gpd.GeoDataFrame): _description_
        stats_group (dict): Statistics based on group
            * Key is the group name, must match the `HOSPITALS` `group` value
            * Valid value dict keys (group-level properties:)
            `workforce`, `beds`, `inpatient`, `outpatient`

    Returns:
    """
    hp_gdf = hp_gdf.copy()

    for group, properties in stats_group.items():
        for prop, group_prop_val in properties.items():
            if prop not in hp_gdf.columns:
                print(f"ERROR: {prop} not in hp_gdf")

            if prop != "workforce":
                # only care about workforce here
                continue

            hp_gdf_group = hp_gdf[hp_gdf["group"] == group]
            hp_gdf_not_group = hp_gdf[hp_gdf["group"] != group]
            hp_gdf_not_group_with_wforce = hp_gdf_not_group[
                (~hp_gdf_not_group["workforce"].isna())
                & (~hp_gdf_not_group["beds"].isna())
            ]

            # print(hp_gdf_not_group_with_wforce[["beds", "floorspace"]])

            workforce_predictions = distribute_values_regr(
                hp_gdf_not_group_with_wforce[["beds", "floorspace"]],
                hp_gdf_not_group_with_wforce[["workforce"]],
                hp_gdf_group[["beds", "floorspace"]].fillna(0),
                regr_model=GammaRegressor,
                total_val=group_prop_val,
            )
            # print(workforce_predictions)

            hp_gdf.loc[hp_gdf["group"] == group, "workforce"] = workforce_predictions

            print(hp_gdf_group[["name", "workforce", "beds", "floorspace"]])

    return hp_gdf

health_plots_full_gdf = distribute_workforce_grouped(
    health_plots_full_gdf, STATS_GROUP
)

print(f"\nTotal current calculated workforce: {health_plots_full_gdf["workforce"].sum()}")
health_plots_full_gdf[~health_plots_full_gdf.workforce.isna()].sort_values(
    "workforce", ascending=False
).reset_index(drop=True).head(20)

p:\Code2\1_PROJ_GROUPS\10_TrainGames\Subway Builder\Mapping\.venv\Lib\site-packages\sklearn\utils\validation.py:1365: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


                         name  workforce    beds     floorspace
6      Tan Tock Seng Hospital        NaN  1700.0  412128.379805
7     Khoo Teck Puat Hospital        NaN  1019.0  185731.906567
8          Woodlands Hospital        NaN  1800.0  245437.238286
9  Institute of Mental Health        NaN  1950.0   87101.513711
                               name  workforce   beds    floorspace
15              Gleneagles Hospital        NaN  257.0  40396.201879
16         Mount Elizabeth Hospital        NaN  345.0  71302.550583
17  Mount Elizabeth Novena Hospital        NaN  333.0  92883.531351
18            Parkway East Hospital        NaN  143.0   4311.765964

Total current calculated workforce: 77224.0


p:\Code2\1_PROJ_GROUPS\10_TrainGames\Subway Builder\Mapping\.venv\Lib\site-packages\sklearn\utils\validation.py:1365: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


,name,workforce,beds,inpatient,outpatient,group,healthcare_type,floorspace,area,geometry,centroid
0,Singapore General Hospital,10725.000000,1931.0,NaN,NaN,SH,acute,402597.095485,121690.291700,"POLYGON ((28345.48 29364.583, 28345.466 29364....",POINT (28153.821 29115.833)
1,National University Hospital,10700.000000,1700.0,NaN,NaN,NUHS,acute,240995.791972,165680.830591,"POLYGON ((22670.46 30693.927, 22670.285 30694....",POINT (22422.679 30743.725)
2,Tan Tock Seng Hospital,10364.601714,1700.0,NaN,NaN,NHG,acute,412128.379805,138731.275296,"MULTIPOLYGON (((29590.189 33473.374, 29581.087...",POINT (29516.374 33711.208)
3,Changi General Hospital,7092.000000,1132.0,NaN,NaN,SH,acute,255082.967672,54002.480305,"MULTIPOLYGON (((40990.17 35744.257, 40988.746 ...",POINT (40901.362 35845.971)
4,Woodlands Hospital,6962.308830,1800.0,NaN,NaN,NHG,acute,245437.238286,76723.749501,"POLYGON ((23840.722 45284.235, 23834.816 45286...",POINT (23706.117 45161.249)
5,Sengkang General Hospital,5979.000000,1320.0,NaN,NaN,SH,acute,302192.442428,67291.431531,"POLYGON ((34572.858 41763.457, 34632.583 41742...",POINT (34608.481 41925.392)
6,KK Women's and Children's Hospital,5411.000000,864.0,NaN,NaN,SH,acute,152371.224310,44723.295021,"POLYGON ((29612.679 32542.31, 29613.26 32544.2...",POINT (29509.22 32537.082)
7,SingHealth Outram Specialist Centres,5083.000000,185.0,NaN,NaN,SH,acute,404394.276426,94170.457794,"POLYGON ((28383.872 28919.498, 28386.576 28920...",POINT (28470.511 29120.347)
8,Institute of Mental Health,4977.665746,1950.0,NaN,NaN,NHG,psychiatric,87101.513711,185665.012968,"POLYGON ((33989.678 40568.846, 33987.395 40570...",POINT (33735.827 40494.807)
9,Khoo Teck Puat Hospital,3217.423709,1019.0,NaN,NaN,NHG,acute,185731.906567,44214.755940,"POLYGON ((28458.873 44982.973, 28405.074 44982...",POINT (28533.401 45100.17)


## Distribute Workforce into Remaining Healthcare Institutions

~~First categorize into likely type of healthcare institution, `healthcare_type`, with each type getting a ratio multiplier~~

Currently, only hospitals (`acute` and `commmunity` `healthcare_type`s) will get workforce numbers.
The underlying data for clinics and nursing homes is not clean enough.
Especially with so much remaining workforce to distribute

Types:

- `acute`
  - gets predictions using `beds` and `floorspace`
- `community`
  - gets predictions using `beds` and `floorspace`...? hmmmm, there's so few data points though
  - probably works *well enough* for feature dimension of 2 though :)
  - Sorry, no, because there are so few data points, both types are lumped together for prediction.
  - This does look like an overprediction at the moment.



In [16]:
def distribute_workforce_remaining(
    hp_gdf: gpd.GeoDataFrame, healthcare_types: list[str] = ["acute", "community"]
):
    """
    Both healthcare_types `acute`, `community` are lumped together, beacuse there are
    too few `community` at the moment to calculate properly
    """

    hp_gdf = hp_gdf.copy()

    # for healthcare_type in healthcare_types:
    hp_mask = (
        # (hp_gdf["healthcare_type"] == healthcare_type)
        hp_gdf["healthcare_type"].isin(healthcare_types)
        & hp_gdf["workforce"].isna()
    )
    hp_non_mask = (
        # (hp_gdf["healthcare_type"] == healthcare_type)
        (hp_gdf["healthcare_type"].isin(healthcare_types))
        & ~hp_gdf["workforce"].isna()
        & ~hp_gdf["beds"].isna()
    )

    hp_gdf.loc[hp_mask, "workforce"] = distribute_values_regr(
        hp_gdf[hp_non_mask][["beds", "floorspace"]],
        hp_gdf[hp_non_mask]["workforce"],
        hp_gdf[hp_mask][["beds", "floorspace"]].fillna(0),
        regr_model=GammaRegressor,
    )

    return hp_gdf


health_plots_workforces_gdf = distribute_workforce_remaining(health_plots_full_gdf)

print(
    f"\nTotal current calculated workforce: {health_plots_workforces_gdf["workforce"].sum()}"
)
health_plots_workforces_gdf[~health_plots_workforces_gdf.workforce.isna()].sort_values(
    "workforce", ascending=False
).reset_index(drop=True).head(32)


Total current calculated workforce: 92009.08720135468


,name,workforce,beds,inpatient,outpatient,group,healthcare_type,floorspace,area,geometry,centroid
0,Singapore General Hospital,10725.000000,1931.0,NaN,NaN,SH,acute,402597.095485,121690.291700,"POLYGON ((28345.48 29364.583, 28345.466 29364....",POINT (28153.821 29115.833)
1,National University Hospital,10700.000000,1700.0,NaN,NaN,NUHS,acute,240995.791972,165680.830591,"POLYGON ((22670.46 30693.927, 22670.285 30694....",POINT (22422.679 30743.725)
2,Tan Tock Seng Hospital,10364.601714,1700.0,NaN,NaN,NHG,acute,412128.379805,138731.275296,"MULTIPOLYGON (((29590.189 33473.374, 29581.087...",POINT (29516.374 33711.208)
3,Changi General Hospital,7092.000000,1132.0,NaN,NaN,SH,acute,255082.967672,54002.480305,"MULTIPOLYGON (((40990.17 35744.257, 40988.746 ...",POINT (40901.362 35845.971)
4,Woodlands Hospital,6962.308830,1800.0,NaN,NaN,NHG,acute,245437.238286,76723.749501,"POLYGON ((23840.722 45284.235, 23834.816 45286...",POINT (23706.117 45161.249)
5,Sengkang General Hospital,5979.000000,1320.0,NaN,NaN,SH,acute,302192.442428,67291.431531,"POLYGON ((34572.858 41763.457, 34632.583 41742...",POINT (34608.481 41925.392)
6,KK Women's and Children's Hospital,5411.000000,864.0,NaN,NaN,SH,acute,152371.224310,44723.295021,"POLYGON ((29612.679 32542.31, 29613.26 32544.2...",POINT (29509.22 32537.082)
7,SingHealth Outram Specialist Centres,5083.000000,185.0,NaN,NaN,SH,acute,404394.276426,94170.457794,"POLYGON ((28383.872 28919.498, 28386.576 28920...",POINT (28470.511 29120.347)
8,Institute of Mental Health,4977.665746,1950.0,NaN,NaN,NHG,psychiatric,87101.513711,185665.012968,"POLYGON ((33989.678 40568.846, 33987.395 40570...",POINT (33735.827 40494.807)
9,Ng Teng Fong General Hospital,4174.378941,1100.0,NaN,NaN,NUHS,acute,196179.465985,54031.899088,"MULTIPOLYGON (((18385.195 34926.313, 18384.791...",POINT (18197.418 35121.564)


In [17]:
build_map_point_numeric_and_category(
    health_plots_workforces_gdf,
    numeric_col="workforce",
    category_col="group",
    geometry_col="centroid",
)

## Existing Problems with the Data (yapping)

Using healthcare plots for mapping hospitals is fine, but it falls apart when trying to map smaller institutions,
many of which often reside inside mixed use buildings that won't be zoned healthcare. 
Currently there are only about 120 extra points, while there should be at least 200-300, not even including small
GP clinics. The 138,000 total figure is also likely not a great total to distribute from, seeing as it's not complete.

In the future, I can see adding polyclinics, as that data is pretty structured, but beyond that, it might not be
worth the effort. 

Also, only using buildings, without amenity POIs. For the Singapore OSM data, the vast majority of clinics (even
large polyclinics) are only mapped as `amenity=clinic` points.

This would require a lot more effort, being having to combine OSM amenity POI data with plots and buildings.
I am too lazy to implement that.

So for the time being, healthcare data will consist only of hospitals:

* 20 Acute hospitals
* 1 Psychiatric hospital
* 10 Community hospitals, with some of the public ones merged into the campuses
  * in the future this could be separated out by having lists in the HOSPITALS definition, then exploded
    in pandas
* Currently tabulates to 92,000, *which is almost definitely an overestimation IMO*


## Export

In [18]:
health_plots_workforces_gdf.columns

Index(['name', 'workforce', 'beds', 'inpatient', 'outpatient', 'group',
       'healthcare_type', 'floorspace', 'area', 'geometry', 'centroid'],
      dtype='str')

In [ ]:
def export_workforce_distr_healthcare(hp_gdf: gpd.GeoDataFrame, fname: str = "healthcare_workforce.parquet"):
    """
    Removes points that don't have workers calculated (in this case, everything that isn't a predefined hospital)

    Keeps all columns, except for `beds`, `inpatient`, `outpatient`:
    * `name`, `workforce` -> `workers`, `group`, `healthcare_type`,
    * `floorspace`, `area`, 
    * `centroid` -> `geometry` and becomes the main geoemtry column, with the original (multi)polygon geometry removed
    
    """
    fpath_intermediate = data_filepaths.path_data_intermediate
    fpath_healthcare_workforce = os.path.join(fpath_intermediate, fname)

    # process
    hp_gdf = hp_gdf.copy()
    crs = hp_gdf.crs
    hp_gdf = hp_gdf.drop(columns=["beds", "inpatient", "outpatient"])
    hp_gdf = hp_gdf.rename(columns={"workforce": "workers"})

    hp_gdf = hp_gdf.drop(columns=["geometry"])
    hp_gdf = hp_gdf.rename(columns={"centroid": "geometry"})
    hp_gdf = hp_gdf.set_geometry("geometry").set_crs(crs)

    hp_gdf = hp_gdf[~hp_gdf["workers"].isna()]

    hp_gdf.to_parquet(fpath_healthcare_workforce)

    return hp_gdf

export_workforce_distr_healthcare(
    health_plots_workforces_gdf, "healthcare_workforce_v1.parquet"
)

,name,workers,group,healthcare_type,floorspace,area,geometry
0,Changi General Hospital,7092.000000,SH,acute,255082.967672,54002.480305,POINT (40901.362 35845.971)
1,Sengkang General Hospital,5979.000000,SH,acute,302192.442428,67291.431531,POINT (34608.481 41925.392)
2,KK Women's and Children's Hospital,5411.000000,SH,acute,152371.224310,44723.295021,POINT (29509.22 32537.082)
3,Outram Community Hospital,724.000000,SH,community,120103.250184,30610.435940,POINT (28202.942 28858.316)
4,Singapore General Hospital,10725.000000,SH,acute,402597.095485,121690.291700,POINT (28153.821 29115.833)
5,SingHealth Outram Specialist Centres,5083.000000,SH,acute,404394.276426,94170.457794,POINT (28470.511 29120.347)
6,Tan Tock Seng Hospital,10364.601714,NHG,acute,412128.379805,138731.275296,POINT (29516.374 33711.208)
7,Khoo Teck Puat Hospital,3217.423709,NHG,acute,185731.906567,44214.755940,POINT (28533.401 45100.17)
8,Woodlands Hospital,6962.308830,NHG,acute,245437.238286,76723.749501,POINT (23706.117 45161.249)
9,Institute of Mental Health,4977.665746,NHG,psychiatric,87101.513711,185665.012968,POINT (33735.827 40494.807)


# Visitation Distribution

Yap a bit about the nationwide statistics